# HF download sanity check

Isolates *where* HF downloads break: API reachability -> a single tiny file -> a tiny full model
-> a medium multi-file model -> hf_transfer. Whichever step stalls tells us the failure mode.
Run cells top to bottom; each prints a timing so a stall is obvious.

In [ ]:
import time, torch, transformers, huggingface_hub
print('transformers   ', transformers.__version__)
print('huggingface_hub', huggingface_hub.__version__)
print('torch          ', torch.__version__, '| cuda', torch.cuda.is_available())
import os
print('HF_HOME        ', os.environ.get('HF_HOME', '(default ~/.cache/huggingface)'))
print('HF_TRANSFER    ', os.environ.get('HF_HUB_ENABLE_HF_TRANSFER', '0'))

In [ ]:
# 1) Can we even reach the HF API? (metadata only, no file bytes)
from huggingface_hub import HfApi
t = time.time()
info = HfApi().model_info('prajjwal1/bert-tiny')
print(f'API reachable in {time.time()-t:.1f}s | {info.modelId} | {len(info.siblings)} files')

In [ ]:
# 2) Download a SINGLE tiny file (force a real network fetch, no cache)
from huggingface_hub import hf_hub_download
t = time.time()
p = hf_hub_download('prajjwal1/bert-tiny', 'config.json', force_download=True)
print(f'single-file download OK in {time.time()-t:.1f}s -> {p}')

In [ ]:
# 3) Full tiny model (~17MB): download + load + forward pass end to end
from transformers import AutoTokenizer, AutoModel
t = time.time()
tok = AutoTokenizer.from_pretrained('prajjwal1/bert-tiny')
m = AutoModel.from_pretrained('prajjwal1/bert-tiny')
out = m(**tok('does hugging face work today', return_tensors='pt'))
print(f'tiny model OK in {time.time()-t:.1f}s | output {tuple(out.last_hidden_state.shape)}')

In [ ]:
# 4) Medium multi-file model (~90MB safetensors) — closer to a real download
t = time.time()
tok2 = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
m2 = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
print(f'medium model OK in {time.time()-t:.1f}s')

In [ ]:
# 5) hf_transfer path (Rust downloader) — re-fetch the tiny model fresh to compare
!pip install -q hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download
t = time.time()
p = hf_hub_download('sentence-transformers/all-MiniLM-L6-v2', 'model.safetensors', force_download=True)
print(f'hf_transfer single-file OK in {time.time()-t:.1f}s -> {p}')
print('\nIf 1-4 pass but the 14.5GB RankZephyr stalls, it is large/multi-shard transfer instability,')
print('not HF being down -> use hf_transfer + resume, or a smaller/pre-sharded reranker.')